# 05 · Charts & getting ready for Streamlit 📈

Numbers in a table are useful; a **picture** is instant. In this last notebook
you'll draw charts from your data, then see how the very same code becomes your
**Streamlit dashboard**.

## What you'll learn
- Plotting a column with `.plot()` (uses **matplotlib**)
- Drawing a clean line chart of closing prices
- Computing a **moving average** and plotting it *with* the price
- How each notebook idea maps onto **Streamlit** (`st.line_chart`, widgets)
- Where to go next: the SPEC, the guide, and the `solution` branch

> 🌐 We download data again so this notebook stands alone. You do **not** need to
> install or run Streamlit here — that comes when you build `app.py`.

## 0. Get data

Same one-ticker download form as before — flat columns mean `data["Close"]` is a
single column we can plot directly.

In [ ]:
import yfinance as yf

data = yf.download("AAPL", period="6mo", auto_adjust=True, multi_level_index=False, progress=False)
data.tail()

## 1. The quickest chart — `.plot()`

Any pandas Series or DataFrame has a `.plot()` method. Calling it on the `Close`
column draws the closing price over time — the date index becomes the x-axis
automatically.

In [ ]:
import matplotlib.pyplot as plt

data["Close"].plot(title="AAPL — closing price")
plt.ylabel("Price ($)")
plt.show()

That's it — one line of real charting code. If you don't see a chart, make sure
you ran the cell (Shift + Enter); the plot appears just beneath it.

### 🏋️ Your turn

Plot the **`Volume`** column instead, with the title `"AAPL — daily volume"`.

In [ ]:
# TODO: call .plot(...) on data["Volume"] with a title, then plt.show().

**One way to do it:**

```python
data["Volume"].plot(title="AAPL — daily volume")
plt.ylabel("Shares traded")
plt.show()
```

## 2. A moving average — smoothing the noise

Daily prices jump around. A **moving average** smooths them by averaging the last
N days at each point, so the trend is easier to see. pandas does it in one call:
`.rolling(window).mean()`.

Here's a 20-day moving average (roughly one trading month).

In [ ]:
data["MA20"] = data["Close"].rolling(20).mean()
data[["Close", "MA20"]].tail()      # first 19 MA20 values are blank (NaN) — not enough days yet

The first 19 rows of `MA20` are empty (`NaN`) because a 20-day average needs 20
days of history before it can start. That's expected.

## 3. Price and moving average on one chart

Because `Close` and `MA20` are both columns now, plotting **both together** is as
easy as selecting them and calling `.plot()`. pandas draws one line per column and
adds a legend.

In [ ]:
data[["Close", "MA20"]].plot(title="AAPL — price vs 20-day average")
plt.ylabel("Price ($)")
plt.show()

### 🏋️ Your turn

Add a **50-day** moving average as a new column `MA50`, then plot `Close`, `MA20`
and `MA50` on the same chart. (A longer window = a smoother, slower line.)

In [ ]:
# TODO: create data["MA50"] with rolling(50), then plot the three columns together.

**One way to do it:**

```python
data["MA50"] = data["Close"].rolling(50).mean()
data[["Close", "MA20", "MA50"]].plot(title="AAPL — price with two moving averages")
plt.ylabel("Price ($)")
plt.show()
```

## 4. From notebook to dashboard 🚀

Here's the exciting part: **you've already written the hard bits.** Streamlit just
wraps them so they run in a web page and react to clicks. The logic is identical —
only the *display* line changes.

| In this notebook | In the Streamlit app (`app.py`) |
| ---------------- | ------------------------------- |
| `data = yf.download(ticker, period, ...)` | **exactly the same** |
| `data` (show the table) | `st.dataframe(data)` |
| `data["Close"].plot()` | `st.line_chart(data["Close"])` |
| `ticker = "AAPL"` (hard-coded) | `ticker = st.text_input("Ticker", "AAPL")` |
| `period = "6mo"` (hard-coded) | `period = st.selectbox("Period", ["1mo","6mo","1y","5y"])` |
| `data[data["Close"] > data["Open"]]` | put behind `st.checkbox("Only up days")` |
| `data["Close"].max()` | `st.metric("Highest close", data["Close"].max())` |

**Widgets** are the magic word. `st.text_input(...)` and `st.selectbox(...)` draw
a box on the page and hand back whatever the user chose — as a normal Python
variable. So your filtering and charting code doesn't change at all; it just
receives *user-chosen* values instead of hard-coded ones.

A tiny taste of what `app.py` looks like (you don't run this here — it needs
`streamlit run`). Notice the download keeps the **same options** you've used all
along, so `data["Close"]` stays a single column:

```python
import streamlit as st
import yfinance as yf

st.title("📈 My Finance Dashboard")

ticker = st.text_input("Ticker", "AAPL")               # a text box
period = st.selectbox("Period", ["1mo", "6mo", "1y"])  # a dropdown

data = yf.download(ticker, period=period,
                   auto_adjust=True, multi_level_index=False, progress=False)

if st.checkbox("Only show up days"):                   # a tick box
    data = data[data["Close"] > data["Open"]]          # your filter from notebook 04!

st.metric("Latest close", data["Close"].iloc[-1])      # a headline number
st.line_chart(data["Close"])                           # your chart from this notebook
st.dataframe(data)                                     # the table
```

Run the real app with `uv run streamlit run app.py` from the project folder.

## 🎓 You made it — now go build it

You've learned the whole toolkit: Python basics, pandas tables, downloading data,
filtering, and charting. That's genuinely everything the dashboard needs. The rest
is assembling these pieces in `app.py`, one stage at a time.

**Where to go next:**
- 📋 **[`../SPEC.md`](../SPEC.md)** — the job: build the dashboard stage by stage.
- 📖 **[`../guides/05-building-the-dashboard.md`](../guides/05-building-the-dashboard.md)**
  — the friendly walkthrough for turning these ideas into `app.py`.
- 💡 **The `solution` branch** — a complete working version to peek at *after* you've
  tried: `git switch solution`. There's no single right answer; compare and learn.

Now open `app.py` and start on **Stage 1**. You've got this. 🎉